In [44]:
from sklearn.linear_model import LinearRegression, SGDRegressor, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, BaggingRegressor
import numpy as np
from sklearn.model_selection import train_test_split
from code_files.data_preperation import prepare_for_train
from code_files.train import train
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV


In [45]:
# Load Dataset
df_amazon = pd.read_csv("dataset/eda_amazon_sales_report.csv")
df_amazon.info()
df_amazon.columns
df_amazon.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 117123 entries, 0 to 117122
Data columns (total 24 columns):
 #   Column                               Non-Null Count   Dtype  
---  ------                               --------------   -----  
 0   Unnamed: 0                           117123 non-null  int64  
 1   Size                                 117123 non-null  int64  
 2   Qty                                  117123 non-null  int64  
 3   Amount                               117123 non-null  float64
 4   promotion-ids                        117123 non-null  int64  
 5   B2B                                  117123 non-null  int64  
 6   Status_Cancelled                     117123 non-null  bool   
 7   Status_Shipped                       117123 non-null  bool   
 8   Status_Shipped - Delivered to Buyer  117123 non-null  bool   
 9   Fulfilment_Amazon                    117123 non-null  bool   
 10  Fulfilment_Merchant                  117123 non-null  bool   
 11  ship-service-

,Unnamed: 0,Size,Qty,Amount,promotion-ids,B2B,Status_Cancelled,Status_Shipped,Status_Shipped - Delivered to Buyer,Fulfilment_Amazon,...,Category_Bottom,Category_Dupatta,Category_Ethnic Dress,Category_Saree,Category_Set,Category_Top,Category_Western Dress,Category_kurta,Month,Day
0,0,2,0,647.62,0,0,True,False,False,False,...,False,False,False,False,True,False,False,False,4,30
1,1,7,1,406.00,1,0,False,False,True,False,...,False,False,False,False,False,False,False,True,4,30
2,2,5,1,329.00,1,1,False,True,False,True,...,False,False,False,False,False,False,False,True,4,30
3,3,4,0,753.33,0,0,True,False,False,False,...,False,False,False,False,False,False,True,False,4,30
4,4,7,1,574.00,0,0,False,True,False,True,...,False,False,False,False,False,True,False,False,4,30


In [46]:
# split  data
dftrain, dfdev = train_test_split(df_amazon, test_size=0.1, random_state=42)
Xtrain, ytrain, Xdev, ydev = prepare_for_train(dftrain, dfdev)



In [47]:
# Grid Search for Bagging
bagging_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_samples': [0.5, 0.7, 1.0],
    'max_features': [0.5, 0.7, 1.0],
    'bootstrap': [True],
    'bootstrap_features': [True, False],
    'base_estimator': [DecisionTreeRegressor(max_depth=8)]
}

bagging_model_grid = BaggingRegressor(random_state=42)
bagging_grid = GridSearchCV(
    estimator=bagging_model_grid,
    param_grid=bagging_param_grid,
    cv=5,
    n_jobs=-1,
    verbose=2,
    scoring='neg_mean_absolute_percentage_error'
)

# Fit grid search
bagging_grid.fit(Xtrain, ytrain)

print("\nBest Bagging Parameters:", bagging_grid.best_params_)
print("Best MAPE Score:", -bagging_grid.best_score_)

# Use best model for predictions and metrics
bagging_model = bagging_grid.best_estimator_
y_pred = bagging_model.predict(Xdev)

# Calculate metrics
mae = mean_absolute_error(ydev, y_pred)
rmse = np.sqrt(mean_squared_error(ydev, y_pred))
r2 = r2_score(ydev, y_pred)

print("\nBagging Results:")
print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2 Score: {r2:.2f}")

Fitting 5 folds for each of 54 candidates, totalling 270 fits

Best Bagging Parameters: {'base_estimator': DecisionTreeRegressor(max_depth=8), 'bootstrap': True, 'bootstrap_features': False, 'max_features': 1.0, 'max_samples': 1.0, 'n_estimators': 200}
Best MAPE Score: 5.287072168168911e+16

Bagging Results:
MAE: 211.20
RMSE: 275.11
R2 Score: 0.05
[CV] END base_estimator=DecisionTreeRegressor(max_depth=8), bootstrap=True, bootstrap_features=True, max_features=0.5, max_samples=0.5, n_estimators=100; total time=   5.4s
[CV] END base_estimator=DecisionTreeRegressor(max_depth=8), bootstrap=True, bootstrap_features=True, max_features=0.5, max_samples=1.0, n_estimators=300; total time=  29.2s
[CV] END base_estimator=DecisionTreeRegressor(max_depth=8), bootstrap=True, bootstrap_features=True, max_features=1.0, max_samples=0.5, n_estimators=200; total time=  27.0s
[CV] END base_estimator=DecisionTreeRegressor(max_depth=8), bootstrap=True, bootstrap_features=False, max_features=0.5, max_samples